# 02_feature.ipynb — Preprocessing and Feature Extraction

**Inputs:** `../data/SC/SC{sid:02d}.npz` (Phase 0 output)  
**Outputs:**
- `../data/SC_filtered/SC{sid:02d}.npz` — filtered epochs + EMG RMS
- `../data/features/feature_matrix.npz` — pooled X, y, subject_ids
- `../data/features/feature_names.txt` — one name per line

**Phase 2 (filtering):** EEG 0.5-40 Hz, EOG 0.5-12 Hz, EMG 10-45 Hz + RMS envelope.  
**Phase 3 (features):** spectral band powers (incl. gamma for RQ2), SEF95, delta/sigma ratio, time-domain, Hjorth, EOG, EMG.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from src.preprocess import filter_epochs
from src.features   import extract_features

SC_DIR   = Path('../data/SC')
FILT_DIR = Path('../data/SC_filtered')
FEAT_DIR = Path('../data/features')
FILT_DIR.mkdir(parents=True, exist_ok=True)
FEAT_DIR.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv('../data/metadata.csv')
subjects = meta['subject'].tolist()
print(f'Subjects to process: {len(subjects)}')

Subjects to process: 76


## Phase 2 — Filter epochs

Resume-safe: skips subjects already written to `SC_filtered/`.

In [2]:
for sid in subjects:
    out = FILT_DIR / f'SC{sid:02d}.npz'
    if out.exists():
        continue
    raw = np.load(SC_DIR / f'SC{sid:02d}.npz')
    epochs, labels, artifact_mask = raw['epochs'], raw['labels'], raw['artifact_mask']
    filtered, emg_rms = filter_epochs(epochs)
    np.savez_compressed(out, epochs=filtered, labels=labels,
                        artifact_mask=artifact_mask, emg_rms=emg_rms)
    print(f'  filtered SC{sid:02d} — {len(labels)} epochs')

print('Phase 2 complete.')

Phase 2 complete.


## Phase 3 — Extract features

Artifact-flagged epochs are excluded here before building the pooled matrix.
Subject IDs are stored alongside X and y for LOSO.

In [5]:
Xs, ys, sids = [], [], []
feat_names   = None

for sid in subjects:
    arr  = np.load(FILT_DIR / f'SC{sid:02d}.npz')
    mask = ~arr['artifact_mask']          # clean epochs only
    eps  = arr['epochs'][mask]
    labs = arr['labels'][mask]
    emg  = arr['emg_rms'][mask]

    if len(labs) < 40:                   # < 20 min of clean data — skip
        print(f'  [SKIP] SC{sid:02d}: only {len(labs)} clean epochs')
        continue

    X_sub, names = extract_features(eps, emg)
    if feat_names is None:
        feat_names = names

    Xs.append(X_sub)
    ys.append(labs)
    sids.append(np.full(len(labs), sid, dtype=np.int32))

X   = np.vstack(Xs)
y   = np.concatenate(ys)
sid_arr = np.concatenate(sids)

print(f'Feature matrix: {X.shape}  |  n_features={len(feat_names)}')
print(f'Subjects retained: {np.unique(sid_arr).size}')

  [SKIP] SC06: only 25 clean epochs
  [SKIP] SC08: only 28 clean epochs
  [SKIP] SC27: only 20 clean epochs
  [SKIP] SC34: only 38 clean epochs
  [SKIP] SC43: only 38 clean epochs
  [SKIP] SC72: only 39 clean epochs
  [SKIP] SC82: only 38 clean epochs
Feature matrix: (6452, 47)  |  n_features=47
Subjects retained: 69


## Save

In [6]:
np.savez_compressed(FEAT_DIR / 'feature_matrix.npz',
                    X=X, y=y, subject_ids=sid_arr)

with open(FEAT_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feat_names))

print('Saved feature_matrix.npz')
print('Saved feature_names.txt')
print(f'Stage counts: { {n: int((y==i).sum()) for i,n in enumerate(["Wake","N1","N2","N3","REM"])} }')

Saved feature_matrix.npz
Saved feature_names.txt
Stage counts: {'Wake': 684, 'N1': 2048, 'N2': 2272, 'N3': 857, 'REM': 591}
